# HypoGuardian AI: Google Colab Model Training Notebook
### End-to-End Training for 30-Minute Hypoglycemia Early Warning with LSTM

This standalone notebook allows you to train the HypoGuardian LSTM model in Google Colab (with free GPU acceleration), evaluate it on held-out test data, and download the trained model and scaler to plug directly into your local Streamlit app.

---

### Step 1: Install & Import Dependencies

In [ ]:
!pip install numpy pandas scikit-learn tensorflow joblib matplotlib

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import json
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve, roc_curve, auc
)
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print('TensorFlow Version:', tf.__version__)

### Step 2: Acquire or Generate Continuous Glucose Monitoring Data
You can upload your own OhioT1DM research CSV/XML or generate the synthetic benchmark cohort below.

In [ ]:
def generate_patient_cgm_trace(patient_id, start_time, days=7, sampling_min=5, seed=42):
    rng = np.random.RandomState(seed)
    total_steps = int((days * 24 * 60) / sampling_min)
    timestamps = [start_time + timedelta(minutes=i * sampling_min) for i in range(total_steps)]
    glucose_values = np.zeros(total_steps)
    current_glucose = 115.0 + rng.normal(0, 5)

    for i, t in enumerate(timestamps):
        hour = t.hour + t.minute / 60.0
        circadian_effect = -10.0 * np.cos((hour - 4.0) * (2 * np.pi / 24.0))
        meal_effect = 0.0
        if 8.0 <= hour <= 9.5:
            meal_effect += 35.0 * np.sin((hour - 8.0) / 1.5 * np.pi)
        elif 13.0 <= hour <= 14.5:
            meal_effect += 40.0 * np.sin((hour - 13.0) / 1.5 * np.pi)
        elif 19.5 <= hour <= 21.0:
            meal_effect += 45.0 * np.sin((hour - 19.5) / 1.5 * np.pi)

        hypo_dip = 0.0
        day_idx = i // (24 * 60 // sampling_min)
        if (day_idx % 2 == 1 and 2.0 <= hour <= 4.0) or (day_idx % 3 == 0 and 16.0 <= hour <= 17.5):
            hypo_dip = -45.0 * np.sin(np.pi * (hour - (2.0 if hour < 5 else 16.0)) / 2.0)

        target = 115.0 + circadian_effect + meal_effect + hypo_dip
        drift = 0.08 * (target - current_glucose)
        noise = rng.normal(0, 2.5)
        current_glucose += drift + noise
        current_glucose = float(np.clip(current_glucose, 35.0, 400.0))
        glucose_values[i] = round(current_glucose, 1)

    return pd.DataFrame({
        'patient_id': patient_id,
        'timestamp': [t.strftime('%Y-%m-%d %H:%M:%S') for t in timestamps],
        'glucose': glucose_values
    })

base_time = datetime(2026, 3, 1, 0, 0, 0)
df_list = [
    generate_patient_cgm_trace('P01', base_time, days=7, seed=101),
    generate_patient_cgm_trace('P02', base_time, days=7, seed=202),
    generate_patient_cgm_trace('P03', base_time, days=7, seed=303),
]
df = pd.concat(df_list, ignore_index=True)
print(f'Total cohort readings: {len(df):,}')
df.head()

### Step 3: Preprocessing & Time-Series Sequence Formulation
- 12 Past Readings = ~55 minutes of history
- 6 Future Readings = 30 minutes forecast horizon
- Target: $y = 1$ if $\min(G_{t+1 \dots t+6}) < 70\text{ mg/dL}$, else $0$

In [ ]:
def create_sequences(df, seq_len=12, horizon=6, hypo_thresh=70.0, max_gap_min=15):
    X_list, y_list = [], []
    for pid, group in df.groupby('patient_id'):
        group = group.sort_values(by='timestamp').reset_index(drop=True)
        ts = pd.to_datetime(group['timestamp']).values
        vals = group['glucose'].values.astype(np.float32)
        total_win = seq_len + horizon
        for i in range(len(vals) - total_win + 1):
            w_ts = ts[i : i + total_win]
            w_vals = vals[i : i + total_win]
            diffs = np.diff(w_ts).astype('timedelta64[s]').astype(float) / 60.0
            if np.any(diffs > max_gap_min) or np.any(diffs <= 0):
                continue
            X_list.append(w_vals[:seq_len].reshape(seq_len, 1))
            y_list.append(1 if np.min(w_vals[seq_len:]) < hypo_thresh else 0)
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int32)

X, y = create_sequences(df)
print(f'Constructed sequences: X shape = {X.shape}, y shape = {y.shape}')
print(f'Positive hypoglycemia labels: {np.sum(y == 1):,} / {len(y):,} ({np.mean(y):.2%})')

### Step 4: Chronological Train / Val / Test Split & Scaler Fitting
Zero leakage: The scaler is fitted strictly on `X_train`.

In [ ]:
N = len(X)
train_end = int(N * 0.70)
val_end = int(N * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.reshape(-1, 1)).reshape(X_train.shape)
X_val_scaled = scaler.transform(X_val.reshape(-1, 1)).reshape(X_val.shape)
X_test_scaled = scaler.transform(X_test.reshape(-1, 1)).reshape(X_test.shape)

classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = {int(c): float(w) for c, w in zip(classes, weights)}
print(f'Train shape: {X_train_scaled.shape}, Val shape: {X_val_scaled.shape}, Test shape: {X_test_scaled.shape}')
print(f'Class weights: {class_weights}')

### Step 5: Define & Train LSTM Neural Network

In [ ]:
model = models.Sequential([
    layers.Input(shape=(12, 1)),
    layers.LSTM(64, return_sequences=False),
    layers.Dropout(0.30),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.20),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')]
)
model.summary()

cb = [
    callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]

history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=35, batch_size=32,
    class_weight=class_weights,
    callbacks=cb
)

### Step 6: Held-Out Test Set Evaluation

In [ ]:
y_preds_prob = model.predict(X_test_scaled).ravel()
y_pred_binary = (y_preds_prob >= 0.50).astype(int)

cm = confusion_matrix(y_test, y_pred_binary)
print('Test Confusion Matrix:\n', cm)
print(f'Test Accuracy:     {accuracy_score(y_test, y_pred_binary):.4f}')
print(f'Sensitivity (Rec): {recall_score(y_test, y_pred_binary, zero_division=0):.4f}')
print(f'Specificity:       {cm[0,0] / (cm[0,0] + cm[0,1]):.4f}')
print(f'ROC-AUC:           {roc_auc_score(y_test, y_preds_prob):.4f}')

### Step 7: Export & Download Artifacts for Local Streamlit App
Run this cell in Google Colab to download `hypo_lstm_model.keras` and `glucose_scaler.pkl` directly to your local computer.

In [ ]:
model.save('hypo_lstm_model.keras')
joblib.dump(scaler, 'glucose_scaler.pkl')

# Google Colab direct download helper
try:
    from google.colab import files
    files.download('hypo_lstm_model.keras')
    files.download('glucose_scaler.pkl')
    print('Downloaded artifacts successfully! Place them inside HypoGuardian-AI/models/')
except ImportError:
    print('Local environment detected; files saved to disk.')